# Tokenizer Pipeline (Sharded)
1. Train or load BPE tokenizer (samples random files from all `data_*` folders)
2. Tokenize each source file into individual shards in `token_shards/`
3. If tokenizer changed, wipes old shards and re-tokenizes everything
4. If tokenizer same, only tokenizes new/changed source files

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

SPARKYLLM = "/content/drive/MyDrive/sparkyllm"
DATA_PATH = os.path.join(SPARKYLLM, "datasets_pretrain")
TOKENIZER_DIR = os.path.join(DATA_PATH, "tokenizer_out")
SHARD_DIR = os.path.join(SPARKYLLM, "token_shards")

os.makedirs(TOKENIZER_DIR, exist_ok=True)
os.makedirs(SHARD_DIR, exist_ok=True)

# Discover all data_* folders and their .txt files
all_source_files = []
data_dirs = sorted([
    d for d in os.listdir(DATA_PATH)
    if d.startswith("data_") and os.path.isdir(os.path.join(DATA_PATH, d))
])

for dirname in data_dirs:
    dirpath = os.path.join(DATA_PATH, dirname)
    files = sorted(f for f in os.listdir(dirpath) if f.endswith(".txt"))
    for f in files:
        all_source_files.append(os.path.join(dirpath, f))
    print(f"  {dirname}: {len(files)} files")

total_size = sum(os.path.getsize(f) for f in all_source_files)
print(f"\nTotal: {len(all_source_files)} source files, {total_size / 1024 / 1024 / 1024:.1f} GB")

In [ ]:
import json
import time
import random
import hashlib
import shutil
import numpy as np
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

# ================= CONFIGURATION =================
TRAIN_NEW_TOKENIZER = False  # Set True to retrain

TOKENIZER_PATH = os.path.join(TOKENIZER_DIR, "tokenizer.json")
SHARD_MANIFEST_PATH = os.path.join(SHARD_DIR, "shard_manifest.json")
META_PATH = os.path.join(SHARD_DIR, "meta.json")
TOK_ID_PATH = os.path.join(SHARD_DIR, "tokenization_id.txt")

VOCAB_SIZE = 32000
TRAIN_SIZE_BYTES = 500 * 1024 * 1024  # 500 MB subset for vocab training
ENCODE_BATCH_SIZE = 10_000

EOT_TOKEN = "<|endoftext|>"
UNK_TOKEN = "<|unk|>"

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

def save_meta(path, shard_manifest, vocab_size, eot_id, unk_id, tok_id):
    """Save meta.json — called after every shard so training can start early."""
    total_tokens = sum(s["tokens"] for s in shard_manifest.get("shards", []))
    total_shards = len(shard_manifest.get("shards", []))
    meta = {
        "num_tokens": total_tokens,
        "num_shards": total_shards,
        "vocab_size": vocab_size,
        "eot_id": eot_id,
        "unk_id": unk_id,
        "tokenizer": "byte-level-bpe",
        "tokenization_id": tok_id,
    }
    with open(path, "w") as f:
        json.dump(meta, f, indent=2)

# ================= STEP 1: TRAIN OR LOAD TOKENIZER =================
if TRAIN_NEW_TOKENIZER:
    subset_path = os.path.join(TOKENIZER_DIR, "bpe_subset.txt")
    print(f"[1/2] Building vocab training subset ({TRAIN_SIZE_BYTES / 1024 / 1024:.0f} MB)...")
    print(f"  Sampling random files from {len(all_source_files)} source files...")

    random.seed(42)
    sampled = all_source_files.copy()
    random.shuffle(sampled)

    current_bytes = 0
    t0 = time.time()
    with open(subset_path, "w", encoding="utf-8") as out_f:
        for src_file in sampled:
            if current_bytes >= TRAIN_SIZE_BYTES:
                break
            with open(src_file, "r", encoding="utf-8", errors="ignore") as in_f:
                for line in in_f:
                    out_f.write(line)
                    current_bytes += len(line.encode("utf-8"))
                    if current_bytes >= TRAIN_SIZE_BYTES:
                        break
    print(f"  Subset: {current_bytes / 1024 / 1024:.0f} MB in {time.time() - t0:.1f}s")

    print(f"  Training Byte-Level BPE tokenizer (vocab={VOCAB_SIZE})...")
    t0 = time.time()
    tokenizer = Tokenizer(BPE(unk_token=UNK_TOKEN, dropout=None))
    tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
    tokenizer.decoder = ByteLevelDecoder()

    trainer = BpeTrainer(
        vocab_size=VOCAB_SIZE,
        min_frequency=2,
        special_tokens=[EOT_TOKEN, UNK_TOKEN],
    )
    tokenizer.train([subset_path], trainer)
    tokenizer.save(TOKENIZER_PATH)
    print(f"  Done in {time.time() - t0:.1f}s -> {TOKENIZER_PATH}")
else:
    print(f"[1/2] Loading existing tokenizer from {TOKENIZER_PATH}...")
    tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

vocab_size_actual = tokenizer.get_vocab_size()
eot_id = tokenizer.token_to_id(EOT_TOKEN)
unk_id = tokenizer.token_to_id(UNK_TOKEN)

assert eot_id is not None, "EOT token missing"
assert unk_id is not None, "UNK token missing"

print(f"  Vocab size: {vocab_size_actual} | EOT: {eot_id} | UNK: {unk_id}")

# ================= STEP 2: CHECK TOKENIZATION_ID =================
current_tok_id = file_hash(TOKENIZER_PATH)[:16]
print(f"  tokenization_id: {current_tok_id}")

old_tok_id = None
if os.path.exists(TOK_ID_PATH):
    with open(TOK_ID_PATH, "r") as f:
        old_tok_id = f.read().strip()

if old_tok_id and old_tok_id != current_tok_id:
    print(f"\n  TOKENIZER CHANGED ({old_tok_id} -> {current_tok_id})")
    print(f"  Wiping all old shards...")
    for f in os.listdir(SHARD_DIR):
        os.remove(os.path.join(SHARD_DIR, f))
    os.makedirs(SHARD_DIR, exist_ok=True)
    print(f"  Done. All shards will be regenerated.")
elif old_tok_id:
    print(f"  Tokenizer unchanged. Will only tokenize new/changed files.")

with open(TOK_ID_PATH, "w") as f:
    f.write(current_tok_id)

# ================= STEP 3: LOAD EXISTING SHARD MANIFEST =================
shard_manifest = {"tokenization_id": current_tok_id, "shards": []}
if os.path.exists(SHARD_MANIFEST_PATH):
    with open(SHARD_MANIFEST_PATH, "r") as f:
        shard_manifest = json.load(f)

existing_shards = {s["source"]: s for s in shard_manifest.get("shards", [])}

# ================= STEP 4: TOKENIZE NEW/CHANGED FILES =================
print(f"\n[2/2] Tokenizing source files into shards...")

shard_idx = len(shard_manifest.get("shards", []))
new_count = 0
skip_count = 0
total_new_tokens = 0
t0 = time.time()

for file_num, src_path in enumerate(all_source_files):
    src_hash = None
    if src_path in existing_shards:
        old_entry = existing_shards[src_path]
        shard_path = os.path.join(SHARD_DIR, old_entry["shard"])
        if os.path.exists(shard_path):
            current_size = os.path.getsize(src_path)
            if current_size == old_entry.get("source_size", -1):
                skip_count += 1
                continue
            src_hash = file_hash(src_path)[:16]
            if src_hash == old_entry.get("source_hash"):
                skip_count += 1
                continue

    if src_hash is None:
        src_hash = file_hash(src_path)[:16]

    src_name = os.path.basename(src_path)
    src_size = os.path.getsize(src_path)
    src_mb = src_size / 1024 / 1024
    shard_name = f"shard_{shard_idx:04d}.bin"
    shard_path = os.path.join(SHARD_DIR, shard_name)

    print(f"  [{file_num+1}/{len(all_source_files)}] {src_name} ({src_mb:.1f} MB)...", end=" ", flush=True)

    token_count = 0
    doc_count = 0
    batch_lines = []
    batch_markers = []

    with open(shard_path, "wb") as f_bin:
        with open(src_path, "r", encoding="utf-8", errors="ignore") as f_txt:
            for line in f_txt:
                stripped = line.strip()
                if not stripped:
                    continue
                if stripped == EOT_TOKEN:
                    batch_markers.append(("eot",))
                    doc_count += 1
                else:
                    batch_markers.append(("text", len(batch_lines)))
                    batch_lines.append(stripped)

                if len(batch_lines) >= ENCODE_BATCH_SIZE:
                    encoded = tokenizer.encode_batch(batch_lines)
                    buffer = []
                    for marker in batch_markers:
                        if marker[0] == "eot":
                            buffer.append(eot_id)
                        else:
                            buffer.extend(encoded[marker[1]].ids)
                    arr = np.array(buffer, dtype=np.uint32)
                    f_bin.write(arr.tobytes())
                    token_count += len(buffer)
                    batch_lines.clear()
                    batch_markers.clear()

        if batch_lines or batch_markers:
            encoded = tokenizer.encode_batch(batch_lines) if batch_lines else []
            buffer = []
            for marker in batch_markers:
                if marker[0] == "eot":
                    buffer.append(eot_id)
                else:
                    buffer.extend(encoded[marker[1]].ids)
            if buffer:
                arr = np.array(buffer, dtype=np.uint32)
                f_bin.write(arr.tobytes())
                token_count += len(buffer)

    shard_mb = os.path.getsize(shard_path) / 1024 / 1024
    print(f"{token_count:,} tokens ({shard_mb:.1f} MB)")

    entry = {
        "shard": shard_name,
        "source": src_path,
        "source_hash": src_hash,
        "source_size": src_size,
        "tokens": token_count,
        "documents": doc_count,
        "shard_mb": round(shard_mb, 2),
    }

    if src_path in existing_shards:
        old_shard = existing_shards[src_path]["shard"]
        old_path = os.path.join(SHARD_DIR, old_shard)
        if os.path.exists(old_path) and old_shard != shard_name:
            os.remove(old_path)
        shard_manifest["shards"] = [
            entry if s["source"] == src_path else s
            for s in shard_manifest["shards"]
        ]
    else:
        shard_manifest["shards"].append(entry)
        shard_idx += 1

    total_new_tokens += token_count
    new_count += 1

    # Save manifest AND meta after each shard (so training can start early)
    shard_manifest["tokenization_id"] = current_tok_id
    with open(SHARD_MANIFEST_PATH, "w") as f:
        json.dump(shard_manifest, f, indent=2)
    save_meta(META_PATH, shard_manifest, vocab_size_actual, eot_id, unk_id, current_tok_id)

elapsed = time.time() - t0
total_tokens = sum(s["tokens"] for s in shard_manifest["shards"])
total_shards = len(shard_manifest["shards"])

print(f"\nDone in {elapsed:.1f}s!")
print(f"  New/updated: {new_count} files ({total_new_tokens:,} tokens)")
print(f"  Skipped:     {skip_count} files (unchanged)")
print(f"  Total:       {total_shards} shards, {total_tokens:,} tokens")
print(f"  Meta:        {META_PATH}")

In [ ]:
# ================= SAVE MANIFEST =================
import datetime

MANIFEST_DIR = os.path.join(SPARKYLLM, "manifests")
os.makedirs(MANIFEST_DIR, exist_ok=True)

manifest = {
    "stage": "tokenization",
    "created": datetime.datetime.now().isoformat(),
    "tokenization_id": current_tok_id,
    "tokenizer": {
        "path": TOKENIZER_PATH,
        "vocab_size": vocab_size_actual,
        "type": "byte-level-bpe",
    },
    "config": {
        "vocab_size_requested": VOCAB_SIZE,
        "trained_new": TRAIN_NEW_TOKENIZER,
        "eot_id": eot_id,
        "unk_id": unk_id,
        "train_subset_mb": TRAIN_SIZE_BYTES / 1024 / 1024,
    },
    "shards": {
        "dir": SHARD_DIR,
        "num_shards": total_shards,
        "total_tokens": total_tokens,
    },
    "sources": {
        "num_data_dirs": len(data_dirs),
        "num_source_files": len(all_source_files),
        "data_dirs": data_dirs,
    },
}

manifest_path = os.path.join(MANIFEST_DIR, "tokenization_latest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\nManifest saved: {manifest_path}")
print(f"  tokenization_id: {current_tok_id}")
print(f"  {total_shards} shards, {total_tokens:,} tokens")